# 1. Paraphrasing test set.

We have already figured out how to paraphrase in the previous ipynb, so this is a trimmed down version of that one, pasted here, solely for paraphrasing the test set.

______________

In [1]:
from openai import OpenAI
import pandas as pd
import numpy as np
import getpass
import time
import os

In [2]:
# use getpass to pass api key over so as not to inclue here
API_KEY = getpass.getpass("Enter OpenAI API key: ")

# create client
client = OpenAI(api_key = API_KEY)

print("Open AI client created.")

Enter OpenAI API key:  ········


Open AI client created.


Now need to set up the prompts that will be passed to Open AI for paraphrasing. Seperate prompts for light paraphrase, heavy paraphrase, translation to French and translation from French back to English.

In [3]:
light_para_prompt = """

This is a text paraphrasing task for an MSc project.
There is no malicious intent.
I only need you to make a light paraphrase of the provided text.

Paraphrase the following text lightly.

Make limited changes to the original wording, phrasing and sentence structure.
Replace some words and phrases with alternatives.
Make minor grammatical changes.

Preserve the original meaning, factual information, level of detail, structure 
and ideas as closely as possible.
The result should be a paraphrase that remains very close to the original text.
Do not add new information, remove information, or summarise the text.

The length of the paraphrase should be similar to the original.
DO NOT END THE PARAPHRASE MID-SENTENCE!
MAKE A PARAPHRASE OF THE ENTIRE ORIGINAL TEXT.

Return only the paraphrased text.


TEXT: {text}

"""

NB: There are two lines in the prmopt above:
DO NOT END THE PARAPHRASE MID-SENTENCE!
MAKE A PARAPHRASE OF THE ENTIRE ORIGINAL TEXT.
These were not standard - at the very end of this ipynb I still had one original piece of text that was re-generated as a light paraphrase countless times and each time cut out midway through. In the end I added these two lines to this prompt to try to force the generation.

In [4]:
heavy_para_prompt = """

Paraphrase the following text heavily.

Make substantial changes to the original wording, phrasing and sentence structure.
Make major grammatical changes, and use substantially different vocabulary.
Reorganise sentences and vary how ideas are expressed where appropriate.


Preserve the original meaning, factual information, level of detail, and ideas 
as closely as possible.
The result should be a paraphrase that is substantially different in wording, 
structure and order to the original text whilst preserving its meaning.
Do not add new information, remove information, or summarise the text.

The length of the paraphrase should be similar to the original.


TEXT: {text}

"""

In [5]:
trans_french_prompt = """

Translate the following English text into French.

Create a natural and fluent translation.
Preserve the original meaning, factual information, level of detail, 
and ideas as closely as possible.
Preserve any names, dates, numbers and technical information.

Do not add new information, remove information, or summarise the text.
Keep the length similar to the original.

Return only the translated French text.


TEXT: {text}

"""

In [6]:
trans_english_prompt = """

Translate the following French text into English.

Create a natural and fluent translation.
Preserve the original meaning, factual information, level of detail, 
and ideas as closely as possible.
Preserve any names, dates, numbers and technical information.

Do not add new information, remove information, or summarise the text.
Keep the length similar to the original.

Return only the translated English text.


TEXT: {text}

"""

Function to send the prmopt via API call

In [7]:
# function to send prompt 
def send_prompt(prompt, model):

    # send request to open AI
    response = client.responses.create(
        # specify model
        model = model,
        # send prompt
        input = prompt
    )

    # get response, trim whitespace
    return response.output_text.strip()


In [27]:
csv_file = "test_dataset_master.csv"
df = pd.read_csv(csv_file)
main = df.copy()

In [28]:
# given the time taken to run the api calls for just the initial 20 texts i forsee this full dataset to take a long time, up to 24 hours.
# done want it to crash out and if it does, or if my api credit runs out i dont want to have to start from the start again etc. hence
# why this is much mroe robust than the above code.

In [29]:
# set new output file to contain all original text and info and new paraphrases
output_file = "test_dataset_master_paraphrased.csv"

# have now decided on luna model
model = "gpt-5.6-luna"


# if this does crash, I want to be able to re-run this cell which should check for existing
# data to determine where to carry on from, rather than re-doing old work
if os.path.exists(output_file):

    # read output file
    para_df = pd.read_csv(output_file)
    # convert df to list of dicts
    results = para_df.to_dict("records")
    # find number completed already
    print("Existing results found: ", len(results))

else:

    # if there were none existing this is first time therefore initialise list
    results = []
    print("Starting from beginning.")



# have established where we are startign from, now need to get the ids of any already complete
completed = {
    row["document_id"]
    for row in results
    if pd.notna(row.get("light_paraphrase"))
    and pd.notna(row.get("heavy_paraphrase"))
    and pd.notna(row.get("translation_paraphrase"))
}
print("Documents completed: ", len(completed))


############################################
# Begin looping through dataset
############################################

# now want to loop through whole dataset, initially skipping any rows that have
# a document ID within completed, established above
for i, (_, row) in enumerate(main.iterrows(), start = 1):

    # get id and text data from csv
    document_id = row["document_id"]
    original_text = row["text"]

    # now look for those ids in completed to see what can be skipped
    if document_id in completed:

        print(
            f"[{i}/{len(main)} ]"
            f"{document_id} already completed"
        )

        # if id existed move onto next and do nothing
        continue

    # otherwise we are now beginning to paraphrase those that havne't been done already
    print(f"\n====== Document {i}/{len(main)} ======")
    print("Document: ", document_id)

    
    ############################################
    # Now we move onto the paraphrasing requests
    ############################################

    try:

        # light paraphrase
        print("Light paraphrase...")
        # add original text from df into light paraphrase prompt
        light = send_prompt(
            light_para_prompt.format(text = original_text),
            model
        )

        # Heavy paraphrase
        print("Heavy paraphrase...")
        # add original text from df into heavy paraphrase prompt
        heavy = send_prompt(
            heavy_para_prompt.format(text = original_text),
            model
        )

        # eng to fre translation
        print("Eng > Fre translation...")
        # add original text from df into translation prompt
        french = send_prompt(
            trans_french_prompt.format(text = original_text),
            model
        )

        # fre to eng translation
        print("Fre > Eng translation...")
        # add french text into translation prompt
        english = send_prompt(
            trans_english_prompt.format(text = french),
            model
        )

        # IDs are now already fixed and a crash mid-way through and re-run could result in duplicate IDs
        # this checks and removes first
        results = [
            existing_row
            for existing_row in results
            if existing_row["document_id"] != document_id
        ]        
        
        # store output
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "word_count" : row["word_count"],
            "label" : row["label"],
            "author" : row["author"],
            "text": original_text,
            "light_paraphrase": light,
            "heavy_paraphrase": heavy,
            "french_translation": french,
            "translation_paraphrase": english
        })

        # also add doc id to completed
        completed.add(document_id)
        print("Document complete")


    ############################################
    # Handle API request errors
    ############################################

    # want to state what the error was and still add the row to the document 
    # so that it is logged and we can continue
    except Exception as e:

        print("Error:", e)
        
        # avoide uplicate id as above
        results = [
            existing_row
            for existing_row in results
            if existing_row["document_id"] != document_id
        ]    
        
        # still want to store output to find issue later if necessary
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "word_count" : row["word_count"],
            "label" : row["label"],
            "author" : row["author"],
            "text": original_text,
            "light_paraphrase": None,
            "heavy_paraphrase": None,
            "french_translation": None,
            "translation_paraphrase": None
        })

        print("Failed: ", document_id)


    ############################################
    # Save
    ############################################

    # want to save to the csv after each document rather than waiting 
    # until very end in case it crashes
    para_df = pd.DataFrame(results)
    para_df.to_csv(output_file, index = False)
    print("Results saved")
    # small half second pause
    time.sleep(0.5)

Starting from beginning.
Documents completed:  0

====== Document 1/800 ======
Document:  HU_AMZ_0393
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 2/800 ======
Document:  HU_GUT_0289
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 3/800 ======
Document:  AI_WIK_0040
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 4/800 ======
Document:  HU_WIK_0085
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 5/800 ======
Document:  HU_WIK_0169
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 6/800 ======
Document:  AI_WIK_0137
Light paraphrase...
Hea

______


In [9]:
csv_file = "test_dataset_master_paraphrased.csv"
para_df = pd.read_csv(csv_file)

In [10]:
print(para_df.shape)

(800, 11)


In [11]:
para_df.head(10)

,document_id,source,topic,word_count,label,author,text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase
0,HU_AMZ_0393,Amazon reviews,Food reviews,347.0,0,human,A lot of the reviews/testimonials I see claim ...,Many of the reviews/testimonials I see claim t...,Many of the reviews and testimonials I have re...,Bon nombre des critiques et témoignages que je...,Many of the reviews and testimonials I see cla...
1,HU_GUT_0289,Gutenberg,Mystery,416.0,0,human,More carriages passed through the lodge-gates ...,More carriages passed through the lodge-gates ...,In the first weeks following Squire Denison’s ...,Davantage de voitures franchirent les grilles ...,More cars passed through the lodge gates durin...
2,AI_WIK_0040,Wikipedia,history,410.0,1,AI,The Antikythera mechanism is an ancient Greek ...,The Antikythera mechanism is an ancient Greek ...,Discovered in 1901 among the remains of a sunk...,Le mécanisme d’Anticythère est un ancien dispo...,The Antikythera mechanism is an ancient Greek ...
3,HU_WIK_0085,Wikipedia,History,404.0,0,human,The history of Ethiopia in the Middle Ages rou...,The history of medieval Ethiopia broadly cover...,Medieval Ethiopian history is generally unders...,L’histoire de l’Éthiopie au Moyen Âge s’étend ...,The history of Ethiopia in the Middle Ages ext...
4,HU_WIK_0169,Wikipedia,Natural sciences,419.0,0,human,"Brian Gregory Keating (born September 9, 1971)...","Brian Gregory Keating (born September 9, 1971)...","Brian Gregory Keating, born on September 9, 19...",Brian Gregory Keating (né le 9 septembre 1971)...,"Brian Gregory Keating (born September 9, 1971)..."
5,AI_WIK_0137,Wikipedia,sports_and_recreation,415.0,1,AI,The omnium is a multi-event competition in tra...,The omnium is a multi-event track cycling comp...,Track cycling’s omnium is a combined event tha...,L’omnium est une compétition à épreuves multip...,The omnium is a multi-event competition in tra...
6,AI_WIK_0096,Wikipedia,sports_and_recreation,407.0,1,AI,Orienteering is a navigational sport in which ...,Orienteering is a navigation sport in which pa...,Orienteering is a competitive navigation activ...,La course d’orientation est un sport de naviga...,Orienteering is a navigation sport in which pa...
7,AI_WIK_0389,Wikipedia,engineering_and_technology,395.0,1,AI,Battery thermal management is the control of t...,Battery thermal management involves controllin...,Battery thermal management refers to regulatin...,La gestion thermique des batteries consiste à ...,Battery thermal management involves controllin...
8,HU_RED_0292,Reddit,AskHistorians,416.0,0,human,The intricate details are difficult for me to ...,The intricate details are hard for me to recal...,I cannot reproduce all the finer points from m...,Les détails complexes sont difficiles à me rap...,The complex details are difficult for me to re...
9,HU_GUT_0081,Gutenberg,Science fiction,402.0,0,human,"The soldier had crept from the tunnel, unobser...","The soldier had slipped out of the tunnel, unn...",Norden was so absorbed in tormenting the capti...,"Le soldat s’était glissé hors du tunnel, sans ...",The soldier had slipped out of the tunnel with...


In [12]:
# check for duplicates
duplicates = para_df["document_id"].duplicated().sum()
duplicates

0

Ensure all paraphrases were created.

In [13]:
print(
    para_df[
        [
            "text",
            "light_paraphrase",
            "heavy_paraphrase",
            "french_translation",
            "translation_paraphrase"
        ]
    ].isna().sum()
)

text                      0
light_paraphrase          0
heavy_paraphrase          0
french_translation        0
translation_paraphrase    0
dtype: int64


In [15]:
# check for blanks
check_blank = [
    "text",
    "light_paraphrase",
    "heavy_paraphrase",
    "french_translation",
    "translation_paraphrase"    
]

for column in check_blank:

    blanks = (para_df[column].fillna("").str.strip().eq("").sum())

    print(f"{column} : {blanks} blanks")

text : 0 blanks
light_paraphrase : 0 blanks
heavy_paraphrase : 0 blanks
french_translation : 0 blanks
translation_paraphrase : 0 blanks


Now need to check word count stats for generated paraphrases. We wanted natural text so didn't want to force a word count on the paraphrases but do want to check for any outliers.

In [17]:
# split string in each cell in the original text row to words to calculate
# word count, do for every row in column...
para_df["original_word_count"] = (
    para_df["text"]
    .fillna("")
    .str.split()
    .str.len()
)

# and do for all 5 text columns...
para_df["light_word_count"] = (
    para_df["light_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["heavy_word_count"] = (
    para_df["heavy_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["french_word_count"] = (
    para_df["french_translation"]
    .fillna("")
    .str.split()
    .str.len()
)
para_df["translation_word_count"] = (
    para_df["translation_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

# ...and print it
print(para_df[
    [
        "original_word_count",
        "light_word_count",
        "heavy_word_count",
        "french_word_count",
        "translation_word_count"
    ]
].describe())

       original_word_count  light_word_count  heavy_word_count  \
count           800.000000        800.000000        800.000000   
mean            397.927500        393.282500        427.798750   
std              21.790095         26.084012         41.806524   
min             347.000000        101.000000        322.000000   
25%             382.000000        378.000000        399.000000   
50%             402.000000        395.000000        427.000000   
75%             413.000000        410.000000        457.250000   
max             448.000000        452.000000        553.000000   

       french_word_count  translation_word_count  
count         800.000000              800.000000  
mean          467.173750              409.765000  
std            41.266721               26.531154  
min           355.000000              330.000000  
25%           437.000000              393.000000  
50%           464.000000              411.000000  
75%           495.000000              427.000000

Happy here for the most part - means similar, some high maxs but as this is after paraphrasing it is to be expected. But light paraphrase has a min of 101 which we should double check. Really should double check all those outside of the 350 / 450 range.

NB: in retrospect these limits were poor as due to paraphrasing, translation etc this resulted in a lot of rows, realistically we must accept that there can be a lower or higher word count and instead widen the range to 300-500.

In [18]:
# get those outisde each side of the (new) range
outside_range = para_df[
    (para_df["original_word_count"] < 300) |
    (para_df["original_word_count"] > 500) |

    (para_df["light_word_count"] < 300) |
    (para_df["light_word_count"] > 500) |

    (para_df["heavy_word_count"] < 300) |
    (para_df["heavy_word_count"] > 500) |

    (para_df["translation_word_count"] < 300) |
    (para_df["translation_word_count"] > 500) 
]

# and print
print(
    outside_range[
        [
            "document_id",
            "original_word_count",
            "light_word_count",
            "heavy_word_count",
            "translation_word_count"
        ]
    ]
)

     document_id  original_word_count  light_word_count  heavy_word_count  \
65   AI_RED_0306                  415               414               506   
74   AI_GUT_0336                  423               435               502   
75   AI_RED_0374                  431               432               553   
118  AI_GUT_0123                  431               436               524   
143  AI_WIK_0291                  403               415               508   
147  AI_WIK_0293                  413               409               502   
175  AI_WIK_0054                  448               447               501   
180  AI_GUT_0268                  435               436               524   
187  AI_RED_0202                  422               425               508   
215  AI_GUT_0363                  415               413               517   
218  HU_BBC_0386                  448               442               503   
233  AI_BBC_0199                  413               413               543   

Going to go through some of these to inspect.

In [44]:
# para_df[para_df["document_id"] == "AI_GUT_0336"]["heavy_paraphrase"].iloc[0]

It seems in each case that the paraphrased versions are acceptable and the high / low word counts are solely due to the paraphrasing. There are two notable exceptions: the light paraphrase of HU_BBC_0080 at 101 words and the light paraphrase of HU_BBC_0344 at 273 words.

In [41]:
# print both the original texet and light paraphrase of HU_BBC_0080
print(para_df[para_df["document_id"] == "HU_BBC_0080"]["text"].iloc[0])
print(para_df[para_df["document_id"] == "HU_BBC_0080"]["light_paraphrase"].iloc[0])

Officer Jason Van Dyke is charged with the first-degree murder of 17-year-old Laquan McDonald, who was stopped by police after slashing car tyres. In a Thanksgiving message on Facebook, the president paid tribute to Chicago protesters for keeping the peace. Four people were arrested overnight as demonstrators marched through the city. The protests were not as widespread or as tense as Tuesday night, which began moments after police released the dashcam video showing the incident. Mr Obama said on Facebook: "Like many Americans, I was deeply disturbed by the footage of the fatal shooting of 17-year-old Laquan McDonald. "This Thanksgiving, I ask everybody to keep those who've suffered tragic loss in our thoughts and prayers, and to be thankful for the overwhelming majority of men and women in uniform who protect our communities with honour." McDonald was stopped by police in October 2014 after reports that he had been walking in the road and slashing car tyres. The video shows him holdin

It's clear here that for some reason the paraphrase just cut out mid-way through a sentence. Lets check the other anomaly.

In [20]:
# print both the original texet and light paraphrase of HU_BBC_0344
print(para_df[para_df["document_id"] == "HU_BBC_0344"]["text"].iloc[0])
print(para_df[para_df["document_id"] == "HU_BBC_0344"]["light_paraphrase"].iloc[0])

Divers from the FBI began searching the waters of Seccombe Lake Park, two miles (3km) from the site of the attack. The two suspects are thought to have visited the park on the day of the shooting, an FBI spokesman said. The attack is being investigated as an act of terror, the deadliest in the US since 9/11. Tashfeen Malik and husband Syed Farook opened fire at a workplace event for Farook's colleagues, killing 14 and wounding 22. Who were the attackers? San Bernardino shooting: The victims The couple, who officials say became radicalised some years ago, were both killed in a shoot-out with police hours after the attack. Addressing reporters at the park on Thursday, the FBI declined to specify what the divers were looking for. "We have indications through leads that at some point they came to this park," said David Bowdich, assistant director in charge of the Los Angeles field office of the FBI. US media quoted law enforcement sources as saying they are looking for electronic devices, 

It's clear that both these just cut out midway through and so I'll re-run the prompt generation for each until I get a full version.

In [49]:
# document_id for text to re-prmopt
doc = "HU_BBC_0080"

# get original text from the paraphrase that was cut off midway throuhg
original_text = para_df.loc[
    para_df["document_id"] == doc,
    "text"
].iloc[0]

# re=prompt the paraphrase
new_para = send_prompt(
    light_para_prompt.format(text = original_text),
    model = "gpt-5.6-luna"
)

# check word count
new_para_word_count = len(new_para.split())

print("New word count: ", new_para_word_count)

# and now replace the old broken paraphrase
para_df.loc[
    para_df["document_id"] == doc,
    "light_paraphrase"
] = new_para

# finally print the new paraphrase to check it's acceptable
print("New paraphrase:")
print(
    para_df.loc[
        para_df["document_id"] == doc,
        "light_paraphrase"
    ].iloc[0]
)

New word count:  374
New paraphrase:
Officer Jason Van Dyke faces a first-degree murder charge in the death of 17-year-old Laquan McDonald, who was stopped by police after slashing car tyres. In a Thanksgiving post on Facebook, the president praised Chicago protesters for maintaining the peace. Four people were arrested overnight as demonstrators marched across the city. The protests were neither as widespread nor as tense as those on Tuesday night, which started shortly after police released dashcam footage of the incident. Mr Obama wrote on Facebook: "Like many Americans, I was deeply disturbed by the video of the fatal shooting of 17-year-old Laquan McDonald. "This Thanksgiving, I ask everyone to keep those who have suffered tragic loss in our thoughts and prayers, and to be thankful for the overwhelming majority of men and women in uniform who serve and protect our communities with honour." McDonald was stopped by police in October 2014 following reports that he had been walking in

I had to run that ~ 10 times to get a decent version, odd, and worthy of investigation if I had the time. Whilst this new one appears to end mid-sentence, it actually ends in a list with no ".". Now to do the same for the other poor paraphrase.

In [93]:
# copy paste from above
# document_id for text to re-prmopt
doc = "HU_BBC_0344"

# get original text from the paraphrase that was cut off midway throuhg
original_text = para_df.loc[
    para_df["document_id"] == doc,
    "text"
].iloc[0]

# re=prompt the paraphrase
new_para = send_prompt(
    light_para_prompt.format(text = original_text),
    model = "gpt-5.6-luna"
)

# check word count
new_para_word_count = len(new_para.split())

print("New word count: ", new_para_word_count)

# and now replace the old broken paraphrase
para_df.loc[
    para_df["document_id"] == doc,
    "light_paraphrase"
] = new_para

# finally print the new paraphrase to check it's acceptable
print("New paraphrase:")
print(
    para_df.loc[
        para_df["document_id"] == doc,
        "light_paraphrase"
    ].iloc[0]
)

New word count:  274
New paraphrase:
Divers from the FBI started searching the waters of Seccombe Lake Park, two miles (3km) from the location of the attack. The two suspects are believed to have visited the park on the day of the shooting, an FBI spokesman said. The attack is being treated as an act of terror, and is the deadliest in the US since 9/11. Tashfeen Malik and her husband, Syed Farook, opened fire at a workplace event for Farook’s colleagues, killing 14 people and injuring 22.

Who were the attackers?

San Bernardino shooting: The victims

The couple, who officials say had become radicalised several years ago, were both killed in a shoot-out with police hours after the attack. Speaking to reporters at the park on Thursday, the FBI refused to say exactly what the divers were searching for. “We have indications through leads that at some point they came to this park,” said David Bowdich, assistant director in charge of the FBI’s Los Angeles field office. US media cited law en

I have re-run this countless times, only option is to remove this row.

In [95]:
# copy df without bad id
para_df = para_df[para_df["document_id"] != "HU_BBC_0344"].copy()
# reset the index
para_df = para_df.reset_index(drop = True)
# print shape to confirm
print(para_df.shape)

(799, 16)


Having deleted one human generated BBC piece of text I should delete one BBC AI generated piece of text for equality and fairness.

In [101]:
# create df of only BBC and AI
bbc_ai_texts = para_df[
    (para_df["source"] == "BBC") &
    (para_df["label"] == 1)
]

# randomly choose 1 from this dataset
random_drop = bbc_ai_texts.sample(n = 1, random_state = 1985)

# print it's ID
print(random_drop["document_id"])

456    AI_BBC_0321
Name: document_id, dtype: object


In [103]:
# now remove that ID
# create copy and drop from that DF
para_df = para_df[
    para_df["document_id"] != "AI_BBC_0321"
].copy()

# reset the index
para_df = para_df.reset_index(drop = True)

# print shape to confirm
print(para_df.shape)

(798, 16)


In [104]:
para_df.head()

,document_id,source,topic,word_count,label,author,text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
0,HU_AMZ_0393,Amazon reviews,Food reviews,347.0,0,human,A lot of the reviews/testimonials I see claim ...,Many of the reviews/testimonials I see claim t...,Many of the reviews and testimonials I have re...,Bon nombre des critiques et témoignages que je...,Many of the reviews and testimonials I see cla...,347,348,379,428,367
1,HU_GUT_0289,Gutenberg,Mystery,416.0,0,human,More carriages passed through the lodge-gates ...,More carriages passed through the lodge-gates ...,In the first weeks following Squire Denison’s ...,Davantage de voitures franchirent les grilles ...,More cars passed through the lodge gates durin...,416,397,397,428,430
2,AI_WIK_0040,Wikipedia,history,410.0,1,AI,The Antikythera mechanism is an ancient Greek ...,The Antikythera mechanism is an ancient Greek ...,Discovered in 1901 among the remains of a sunk...,Le mécanisme d’Anticythère est un ancien dispo...,The Antikythera mechanism is an ancient Greek ...,410,415,463,475,432
3,HU_WIK_0085,Wikipedia,History,404.0,0,human,The history of Ethiopia in the Middle Ages rou...,The history of medieval Ethiopia broadly cover...,Medieval Ethiopian history is generally unders...,L’histoire de l’Éthiopie au Moyen Âge s’étend ...,The history of Ethiopia in the Middle Ages ext...,404,386,417,427,420
4,HU_WIK_0169,Wikipedia,Natural sciences,419.0,0,human,"Brian Gregory Keating (born September 9, 1971)...","Brian Gregory Keating (born September 9, 1971)...","Brian Gregory Keating, born on September 9, 19...",Brian Gregory Keating (né le 9 septembre 1971)...,"Brian Gregory Keating (born September 9, 1971)...",419,425,430,456,431


Can now just get rid of unneccesary columns and finally save.

In [105]:
remove_columns = [
    "original_word_count",
    "light_word_count",
    "heavy_word_count",
    "french_word_count",
    "translation_word_count"
]

para_df = para_df.drop(columns = remove_columns)

In [106]:
para_df.head()

,document_id,source,topic,word_count,label,author,text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase
0,HU_AMZ_0393,Amazon reviews,Food reviews,347.0,0,human,A lot of the reviews/testimonials I see claim ...,Many of the reviews/testimonials I see claim t...,Many of the reviews and testimonials I have re...,Bon nombre des critiques et témoignages que je...,Many of the reviews and testimonials I see cla...
1,HU_GUT_0289,Gutenberg,Mystery,416.0,0,human,More carriages passed through the lodge-gates ...,More carriages passed through the lodge-gates ...,In the first weeks following Squire Denison’s ...,Davantage de voitures franchirent les grilles ...,More cars passed through the lodge gates durin...
2,AI_WIK_0040,Wikipedia,history,410.0,1,AI,The Antikythera mechanism is an ancient Greek ...,The Antikythera mechanism is an ancient Greek ...,Discovered in 1901 among the remains of a sunk...,Le mécanisme d’Anticythère est un ancien dispo...,The Antikythera mechanism is an ancient Greek ...
3,HU_WIK_0085,Wikipedia,History,404.0,0,human,The history of Ethiopia in the Middle Ages rou...,The history of medieval Ethiopia broadly cover...,Medieval Ethiopian history is generally unders...,L’histoire de l’Éthiopie au Moyen Âge s’étend ...,The history of Ethiopia in the Middle Ages ext...
4,HU_WIK_0169,Wikipedia,Natural sciences,419.0,0,human,"Brian Gregory Keating (born September 9, 1971)...","Brian Gregory Keating (born September 9, 1971)...","Brian Gregory Keating, born on September 9, 19...",Brian Gregory Keating (né le 9 septembre 1971)...,"Brian Gregory Keating (born September 9, 1971)..."


In [108]:
para_df.to_csv("final_test_dataset.csv", index = False)

____________

# Returning here for objective 5

___________

As per the objective 5 ipynb I need to re-paraphrase a new 300 test set. Going to run helper cells at top then paste main cell again here for output at bottom.

In [8]:
df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_test_300.csv"
)
main = df.copy()

In [10]:
# set new output file to contain all original text and info and new paraphrases
output_file = "final_test_300_paraphrased.csv"

# have now decided on luna model
model = "gpt-5.6-luna"


# if this does crash, I want to be able to re-run this cell which should check for existing
# data to determine where to carry on from, rather than re-doing old work
if os.path.exists(output_file):

    # read output file
    para_df = pd.read_csv(output_file)
    # convert df to list of dicts
    results = para_df.to_dict("records")
    # find number completed already
    print("Existing results found: ", len(results))

else:

    # if there were none existing this is first time therefore initialise list
    results = []
    print("Starting from beginning.")



# have established where we are startign from, now need to get the ids of any already complete
completed = {
    row["document_id"]
    for row in results
    if pd.notna(row.get("light_paraphrase"))
    and pd.notna(row.get("heavy_paraphrase"))
    and pd.notna(row.get("translation_paraphrase"))
}
print("Documents completed: ", len(completed))


############################################
# Begin looping through dataset
############################################

# now want to loop through whole dataset, initially skipping any rows that have
# a document ID within completed, established above
for i, (_, row) in enumerate(main.iterrows(), start = 1):

    # get id and text data from csv
    document_id = row["document_id"]
    original_text = row["text"]

    # now look for those ids in completed to see what can be skipped
    if document_id in completed:

        print(
            f"[{i}/{len(main)} ]"
            f"{document_id} already completed"
        )

        # if id existed move onto next and do nothing
        continue

    # otherwise we are now beginning to paraphrase those that havne't been done already
    print(f"\n====== Document {i}/{len(main)} ======")
    print("Document: ", document_id)

    
    ############################################
    # Now we move onto the paraphrasing requests
    ############################################

    try:

        # light paraphrase
        print("Light paraphrase...")
        # add original text from df into light paraphrase prompt
        light = send_prompt(
            light_para_prompt.format(text = original_text),
            model
        )

        # Heavy paraphrase
        print("Heavy paraphrase...")
        # add original text from df into heavy paraphrase prompt
        heavy = send_prompt(
            heavy_para_prompt.format(text = original_text),
            model
        )

        # eng to fre translation
        print("Eng > Fre translation...")
        # add original text from df into translation prompt
        french = send_prompt(
            trans_french_prompt.format(text = original_text),
            model
        )

        # fre to eng translation
        print("Fre > Eng translation...")
        # add french text into translation prompt
        english = send_prompt(
            trans_english_prompt.format(text = french),
            model
        )

        # IDs are now already fixed and a crash mid-way through and re-run could result in duplicate IDs
        # this checks and removes first
        results = [
            existing_row
            for existing_row in results
            if existing_row["document_id"] != document_id
        ]        
        
        # store output
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "word_count" : row["word_count"],
            "label" : row["label"],
            "author" : row["author"],
            "text": original_text,
            "light_paraphrase": light,
            "heavy_paraphrase": heavy,
            "french_translation": french,
            "translation_paraphrase": english
        })

        # also add doc id to completed
        completed.add(document_id)
        print("Document complete")


    ############################################
    # Handle API request errors
    ############################################

    # want to state what the error was and still add the row to the document 
    # so that it is logged and we can continue
    except Exception as e:

        print("Error:", e)
        
        # avoide uplicate id as above
        results = [
            existing_row
            for existing_row in results
            if existing_row["document_id"] != document_id
        ]    
        
        # still want to store output to find issue later if necessary
        results.append({
            "document_id": document_id,
            "source": row["source"],
            "topic": row["topic"],
            "word_count" : row["word_count"],
            "label" : row["label"],
            "author" : row["author"],
            "text": original_text,
            "light_paraphrase": None,
            "heavy_paraphrase": None,
            "french_translation": None,
            "translation_paraphrase": None
        })

        print("Failed: ", document_id)


    ############################################
    # Save
    ############################################

    # want to save to the csv after each document rather than waiting 
    # until very end in case it crashes
    para_df = pd.DataFrame(results)
    para_df.to_csv(output_file, index = False)
    print("Results saved")
    # small half second pause
    time.sleep(0.5)

Starting from beginning.
Documents completed:  0

====== Document 1/300 ======
Document:  AI_BBC_0050
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 2/300 ======
Document:  HU_WIK_0354
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 3/300 ======
Document:  HU_AMZ_0021
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 4/300 ======
Document:  AI_WIK_0377
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 5/300 ======
Document:  AI_BBC_0162
Light paraphrase...
Heavy paraphrase...
Eng > Fre translation...
Fre > Eng translation...
Document complete
Results saved

====== Document 6/300 ======
Document:  AI_GUT_0060
Light paraphrase...
Hea

___

Now going to run the same checks as above.

In [11]:
print(para_df.shape)

(300, 11)


In [12]:
# check for duplicates
duplicates = para_df["document_id"].duplicated().sum()
duplicates

0

In [13]:
print(
    para_df[
        [
            "text",
            "light_paraphrase",
            "heavy_paraphrase",
            "french_translation",
            "translation_paraphrase"
        ]
    ].isna().sum()
)

text                      0
light_paraphrase          0
heavy_paraphrase          0
french_translation        0
translation_paraphrase    0
dtype: int64


In [14]:
# check for blanks
check_blank = [
    "text",
    "light_paraphrase",
    "heavy_paraphrase",
    "french_translation",
    "translation_paraphrase"    
]

for column in check_blank:

    blanks = (para_df[column].fillna("").str.strip().eq("").sum())

    print(f"{column} : {blanks} blanks")

text : 0 blanks
light_paraphrase : 0 blanks
heavy_paraphrase : 0 blanks
french_translation : 0 blanks
translation_paraphrase : 0 blanks


In [15]:
# split string in each cell in the original text row to words to calculate
# word count, do for every row in column...
para_df["original_word_count"] = (
    para_df["text"]
    .fillna("")
    .str.split()
    .str.len()
)

# and do for all 5 text columns...
para_df["light_word_count"] = (
    para_df["light_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["heavy_word_count"] = (
    para_df["heavy_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

para_df["french_word_count"] = (
    para_df["french_translation"]
    .fillna("")
    .str.split()
    .str.len()
)
para_df["translation_word_count"] = (
    para_df["translation_paraphrase"]
    .fillna("")
    .str.split()
    .str.len()
)

# ...and print it
print(para_df[
    [
        "original_word_count",
        "light_word_count",
        "heavy_word_count",
        "french_word_count",
        "translation_word_count"
    ]
].describe())

       original_word_count  light_word_count  heavy_word_count  \
count           300.000000        300.000000        300.000000   
mean            398.163333        395.676667        423.750000   
std              22.138134         26.552632         43.507515   
min             343.000000        251.000000        308.000000   
25%             383.750000        382.000000        392.000000   
50%             402.000000        398.500000        425.500000   
75%             413.000000        413.250000        452.250000   
max             449.000000        453.000000        550.000000   

       french_word_count  translation_word_count  
count         300.000000              300.000000  
mean          465.506667              408.653333  
std            44.544480               26.621712  
min           361.000000              341.000000  
25%           431.750000              392.000000  
50%           464.000000              413.000000  
75%           500.250000              428.000000

In [16]:
# get those outisde each side of the (new) range
outside_range = para_df[
    (para_df["original_word_count"] < 300) |
    (para_df["original_word_count"] > 500) |

    (para_df["light_word_count"] < 300) |
    (para_df["light_word_count"] > 500) |

    (para_df["heavy_word_count"] < 300) |
    (para_df["heavy_word_count"] > 500) |

    (para_df["translation_word_count"] < 300) |
    (para_df["translation_word_count"] > 500) 
]

# and print
print(
    outside_range[
        [
            "document_id",
            "original_word_count",
            "light_word_count",
            "heavy_word_count",
            "translation_word_count"
        ]
    ]
)

     document_id  original_word_count  light_word_count  heavy_word_count  \
15   AI_RED_0268                  414               415               505   
40   HU_BBC_0375                  357               251               372   
46   AI_GUT_0183                  428               429               513   
66   AI_WIK_0310                  422               421               512   
85   AI_RED_0211                  381               383               550   
94   AI_WIK_0285                  409               405               510   
139  AI_RED_0184                  405               426               518   
187  AI_BBC_0249                  426               429               514   
193  AI_WIK_0376                  434               444               506   
255  AI_RED_0318                  414               429               512   
278  AI_RED_0236                  406               412               504   
298  AI_GUT_0361                  410               413               503   

Issue here again, is that if the word count goes out of range due to paraphrasing, this is the effect of paraphrasing that we are tring to measure. In fact all of these bar one are due to a heavy paraphrase taking the word count higher and as this the affect we are investigating and only counts for 11/300 texts I am happy to leave alone. However, the particularly low word count should be inspected.

In [20]:
para_df[para_df["document_id"] == "HU_BBC_0375"]["light_paraphrase"].iloc[0]

'The 15-year-old, who is currently staying with relatives, has been subjected to an online and in-person harassment campaign. People on Facebook threatened that “they are going to get” her daughter, the 32-year-old woman told US media. The BBC is not naming the mother in order to protect the girl’s privacy. Around 40 people were reportedly watching the livestream, but no one reported the incident to police. The girl has been too afraid to return to the Lawndale neighbourhood where she lives. Her mother says local children have been ringing her doorbell asking for the girl. “This is just disturbing, and to think the kids believe it is funny,” she said, adding that she now wants to leave the crime-ridden west Chicago neighbourhood. “I can’t stay here,” she said on Wednesday after being reunited with her daughter. Chicago police found the girl the day after she went missing and contacted Facebook to get the video removed. The girl has now been discharged from hospital, but has not yet gon

Cut out mid-sentence so will re-run.

In [23]:
# document_id for text to re-prmopt
doc = "HU_BBC_0375"

# get original text from the paraphrase that was cut off midway throuhg
original_text = para_df.loc[
    para_df["document_id"] == doc,
    "text"
].iloc[0]

# re=prompt the paraphrase
new_para = send_prompt(
    light_para_prompt.format(text = original_text),
    model = "gpt-5.6-luna"
)

# check word count
new_para_word_count = len(new_para.split())

print("New word count: ", new_para_word_count)

# and now replace the old broken paraphrase
para_df.loc[
    para_df["document_id"] == doc,
    "light_paraphrase"
] = new_para

# finally print the new paraphrase to check it's acceptable
print("New paraphrase:")
print(
    para_df.loc[
        para_df["document_id"] == doc,
        "light_paraphrase"
    ].iloc[0]
)

New word count:  358
New paraphrase:
The 15-year-old, who is currently staying with relatives, has been subjected to an online and in-person harassment campaign. People on Facebook threatened that “they are going to get” her daughter, the 32-year-old woman told US media. The BBC is not naming the mother in order to protect the girl’s privacy. About 40 people were reportedly watching the livestream, but no one reported the incident to police. The girl has been too afraid to return to the Lawndale neighbourhood where she lives. Her mother says local children have been ringing her doorbell looking for the girl. “This is simply disturbing, and to think the kids believe it is funny,” she said, adding that she now wants to leave the crime-ridden west Chicago neighbourhood. “I can’t remain here,” she said on Wednesday after being reunited with her daughter. Chicago police located the girl a day after she went missing and contacted Facebook to get the video taken down. The girl has now been re

358 meets the minimum threshold and this took multiple attempts. It appears that sometimes the LLM cuts out when detecting violence or certain other types of content as all those that have been cut out are of similar categories.

In [25]:
print(para_df.shape)
(para_df.head(10))

(300, 16)


,document_id,source,topic,word_count,label,author,text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase,original_word_count,light_word_count,heavy_word_count,french_word_count,translation_word_count
0,AI_BBC_0050,BBC,uk,382.0,1,AI,Residents in a coastal town in Norfolk have be...,Residents of a coastal town in Norfolk have be...,People living in the Norfolk seaside town of W...,Les habitants d’une ville côtière du Norfolk o...,Residents of a coastal town in Norfolk have be...,382,383,395,464,394
1,HU_WIK_0354,Wikipedia,Social sciences and society,411.0,0,human,José Manuel Garza Rendón (born 7 December 1952...,José Manuel Garza Rendón (born 7 December 1952...,José Manuel Garza Rendón (born 7 December 1952...,José Manuel Garza Rendón (né le 7 décembre 195...,"José Manuel Garza Rendón (born December 7, 195...",411,423,364,524,425
2,HU_AMZ_0021,Amazon reviews,Food reviews,343.0,0,human,Layla is extremely picky and has left certain ...,Layla is extremely fussy and has even left som...,Layla is exceptionally selective about what sh...,Layla est extrêmement difficile et a simplemen...,Layla is extremely fussy and actually let some...,343,347,382,386,345
3,AI_WIK_0377,Wikipedia,engineering_and_technology,423.0,1,AI,Lithium-ion battery recycling is the recovery ...,Lithium-ion battery recycling involves recover...,Lithium-ion battery recycling involves reclaim...,Le recyclage des batteries lithium-ion consist...,Lithium-ion battery recycling consists of reco...,423,426,466,566,442
4,AI_BBC_0162,BBC,business,401.0,1,AI,A battery recycling plant planned for Teesside...,A battery recycling facility planned for Teess...,Plans for a £240m battery-recycling facility o...,Une usine de recyclage de batteries prévue dan...,A battery recycling plant planned for Teesside...,401,397,478,526,408
5,AI_GUT_0060,Gutenberg,mystery,392.0,1,AI,By the time Eleanor Vane reached the old obser...,By the time Eleanor Vane arrived at the old ob...,When Eleanor Vane arrived at the deserted obse...,Lorsque Eleanor Vane atteignit l’ancien observ...,"When Eleanor Vane reached the old observatory,...",392,400,473,426,392
6,AI_GUT_0233,Gutenberg,english_fiction,433.0,1,AI,By the time Elias Venn reached the old ferry h...,By the time Elias Venn arrived at the old ferr...,When Elias Venn arrived at the deserted ferry ...,Lorsque Elias Venn atteignit l’ancienne maison...,When Elias Venn reached the old ferryman’s hou...,433,426,457,464,434
7,AI_RED_0335,Reddit,ask_historians,364.0,1,AI,I’ve been reading about the Norman conquest of...,I’ve been studying the Norman conquest of Engl...,I have been studying the Norman takeover of En...,Je lisais des textes sur la conquête normande ...,I was reading texts about the Norman conquest ...,364,375,443,463,399
8,AI_RED_0336,Reddit,ask_historians,389.0,1,AI,I’ve been reading about the Salem witch trials...,I’ve been learning about the Salem witch trial...,What continues to strike me about the Salem wi...,Je lis des textes sur les procès de sorcelleri...,I have been reading texts about the Salem witc...,389,387,373,481,426
9,AI_AMZ_0356,Amazon reviews,food_reviews,359.0,1,AI,I bought the 12-pack of Fly By Jing Sichuan Ch...,I purchased the 12-pack of Fly By Jing Sichuan...,After watching Fly By Jing Sichuan Chili Crisp...,J’ai acheté le paquet de 12 pots de Sichuan Ch...,I bought the 12-jar pack of Fly By Jing Sichua...,359,356,369,397,366


Can now drop unneccesary columns.

In [26]:
remove_columns = [
    "original_word_count",
    "light_word_count",
    "heavy_word_count",
    "french_word_count",
    "translation_word_count"
]
para_df = para_df.drop(columns = remove_columns)

In [27]:
para_df.head(10)

,document_id,source,topic,word_count,label,author,text,light_paraphrase,heavy_paraphrase,french_translation,translation_paraphrase
0,AI_BBC_0050,BBC,uk,382.0,1,AI,Residents in a coastal town in Norfolk have be...,Residents of a coastal town in Norfolk have be...,People living in the Norfolk seaside town of W...,Les habitants d’une ville côtière du Norfolk o...,Residents of a coastal town in Norfolk have be...
1,HU_WIK_0354,Wikipedia,Social sciences and society,411.0,0,human,José Manuel Garza Rendón (born 7 December 1952...,José Manuel Garza Rendón (born 7 December 1952...,José Manuel Garza Rendón (born 7 December 1952...,José Manuel Garza Rendón (né le 7 décembre 195...,"José Manuel Garza Rendón (born December 7, 195..."
2,HU_AMZ_0021,Amazon reviews,Food reviews,343.0,0,human,Layla is extremely picky and has left certain ...,Layla is extremely fussy and has even left som...,Layla is exceptionally selective about what sh...,Layla est extrêmement difficile et a simplemen...,Layla is extremely fussy and actually let some...
3,AI_WIK_0377,Wikipedia,engineering_and_technology,423.0,1,AI,Lithium-ion battery recycling is the recovery ...,Lithium-ion battery recycling involves recover...,Lithium-ion battery recycling involves reclaim...,Le recyclage des batteries lithium-ion consist...,Lithium-ion battery recycling consists of reco...
4,AI_BBC_0162,BBC,business,401.0,1,AI,A battery recycling plant planned for Teesside...,A battery recycling facility planned for Teess...,Plans for a £240m battery-recycling facility o...,Une usine de recyclage de batteries prévue dan...,A battery recycling plant planned for Teesside...
5,AI_GUT_0060,Gutenberg,mystery,392.0,1,AI,By the time Eleanor Vane reached the old obser...,By the time Eleanor Vane arrived at the old ob...,When Eleanor Vane arrived at the deserted obse...,Lorsque Eleanor Vane atteignit l’ancien observ...,"When Eleanor Vane reached the old observatory,..."
6,AI_GUT_0233,Gutenberg,english_fiction,433.0,1,AI,By the time Elias Venn reached the old ferry h...,By the time Elias Venn arrived at the old ferr...,When Elias Venn arrived at the deserted ferry ...,Lorsque Elias Venn atteignit l’ancienne maison...,When Elias Venn reached the old ferryman’s hou...
7,AI_RED_0335,Reddit,ask_historians,364.0,1,AI,I’ve been reading about the Norman conquest of...,I’ve been studying the Norman conquest of Engl...,I have been studying the Norman takeover of En...,Je lisais des textes sur la conquête normande ...,I was reading texts about the Norman conquest ...
8,AI_RED_0336,Reddit,ask_historians,389.0,1,AI,I’ve been reading about the Salem witch trials...,I’ve been learning about the Salem witch trial...,What continues to strike me about the Salem wi...,Je lis des textes sur les procès de sorcelleri...,I have been reading texts about the Salem witc...
9,AI_AMZ_0356,Amazon reviews,food_reviews,359.0,1,AI,I bought the 12-pack of Fly By Jing Sichuan Ch...,I purchased the 12-pack of Fly By Jing Sichuan...,After watching Fly By Jing Sichuan Chili Crisp...,J’ai acheté le paquet de 12 pots de Sichuan Ch...,I bought the 12-jar pack of Fly By Jing Sichua...


In [28]:
para_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_test_300_paraphrased.csv", 
    index = False
)